In [ ]:
!pip install -q transformers gradio torch

In [1]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch
import gradio as gr
import warnings

warnings.filterwarnings("ignore")

MODEL_NAME = "ahotrod/electra_large_discriminator_squad2_512"

print("Loading Question Answering model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Device:", device)

Loading Question Answering model...


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda


In [4]:
def chat(user_message, context):
    if not user_message.strip():
        return "Please enter a question."

    if not context.strip():
        return "Please enter a topic or passage first."

    inputs = tokenizer(
        user_message,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_index = torch.argmax(outputs.start_logits, dim=1).item()
    end_index = torch.argmax(outputs.end_logits, dim=1).item()

    if start_index > end_index:
        return "I could not find a clear answer in the given context."

    answer_tokens = inputs["input_ids"][0][start_index:end_index + 1]

    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    ).strip()

    # SQuAD2 models can indicate that no answer exists.
    if (
        not answer
        or start_index == 0
        or end_index >= inputs["input_ids"].shape[1] - 1
    ):
        return "I could not find the answer in the given context."

    return answer

In [5]:
context = """
Artificial intelligence is a technology that enables computers
to perform tasks that normally require human intelligence.
Machine learning is a branch of artificial intelligence.
It allows computers to learn patterns from data.
Computer vision helps computers understand images and videos.
"""

question = "What is machine learning?"

answer = chat(question, context)

print("Question:", question)
print("Answer:", answer)

Question: What is machine learning?
Answer: allows computers to learn patterns from data


In [6]:
context = """
Artificial intelligence helps machines perform intelligent tasks.
Machine learning learns patterns from data.
Deep learning uses neural networks with multiple layers.
Computer vision allows computers to understand images and videos.
Natural language processing helps computers understand human language.
"""

questions = [
    "What is artificial intelligence?",
    "What does machine learning learn from?",
    "What is deep learning?",
    "What does computer vision do?",
    "What is natural language processing?"
]

for question in questions:
    answer = chat(question, context)

    print("Question:", question)
    print("Answer:", answer)
    print("-" * 50)

Question: What is artificial intelligence?
Answer: helps machines perform intelligent tasks
--------------------------------------------------
Question: What does machine learning learn from?
Answer: data
--------------------------------------------------
Question: What is deep learning?
Answer: neural networks with multiple layers
--------------------------------------------------
Question: What does computer vision do?
Answer: allows computers to understand images and videos
--------------------------------------------------
Question: What is natural language processing?
Answer: helps computers understand human language
--------------------------------------------------


In [13]:
import gradio as gr

def chat_app(user_message, context, history):
    history = list(history or [])

    if not user_message.strip():
        return history, ""

    if not context or not context.strip():
        answer = "Please enter a topic or passage first."
    else:
        answer = chat(user_message, context)

    history.append((user_message, answer))
    return history, ""


with gr.Blocks(title="AI Question Answering Chatbot") as app:
    gr.Markdown("# AI Question Answering Chatbot")
    gr.Markdown("Enter a topic, then ask questions about it.")

    context_input = gr.Textbox(
        label="Enter Your Topic or Context",
        lines=6,
        placeholder="Enter information about any topic..."
    )

    chatbox = gr.Chatbot(
        label="Conversation",
        height=350,
        value=[]
    )

    msg_input = gr.Textbox(
        label="Your Question",
        placeholder="Type your question here..."
    )

    with gr.Row():
        send_btn = gr.Button("Ask", variant="primary")
        clear_btn = gr.Button("Clear Chat")

    send_btn.click(
        chat_app,
        inputs=[msg_input, context_input, chatbox],
        outputs=[chatbox, msg_input]
    )

    msg_input.submit(
        chat_app,
        inputs=[msg_input, context_input, chatbox],
        outputs=[chatbox, msg_input]
    )

    clear_btn.click(
        lambda: ([], ""),
        inputs=[],
        outputs=[chatbox, msg_input]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a579372d75ae24e93.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
gr.close_all()
print("Chatbot closed.")

Chatbot closed.
